## Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torchvision import transforms
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

## Loading the Dataset

In [2]:
DATA_DIR = "/kaggle/input/datasets/marccrasto/skin-cancer-dataset"
df = pd.read_csv(f"{DATA_DIR}/df_final.csv")
df.head()

,url,filename,diag,fst,skin_tone_group,partition,label,nine_partition_label,three_partition_label,preprocessed_path
0,https://www.dermaamin.com/site/images/clinical...,ne-de_f2_61_3a96728f,ne-de,2,light,train,neutrophilic dermatoses,inflammatory,non-neoplastic,preprocessed\ne-de_f2_61_3a96728f.jpg
1,https://www.dermaamin.com/site/images/clinical...,lu-er_f1_89_07be6be3,lu-er,1,light,train,lupus erythematosus,inflammatory,non-neoplastic,preprocessed\lu-er_f1_89_07be6be3.jpg
2,http://atlasdermatologico.com.br/img?imageId=4264,me_f0_194_72bcd875,me,0,unknown,train,melanoma,malignant melanoma,malignant,preprocessed\me_f0_194_72bcd875.jpg
3,https://www.dermaamin.com/site/images/clinical...,dr-er_f4_36_dfa97968,dr-er,4,medium,train,drug eruption,inflammatory,non-neoplastic,preprocessed\dr-er_f4_36_dfa97968.jpg
4,http://atlasdermatologico.com.br/img?imageId=8098,ju-xa_f2_118_be506d4a,ju-xa,2,light,train,juvenile xanthogranuloma,inflammatory,non-neoplastic,preprocessed\ju-xa_f2_118_be506d4a.jpg


## Train, Test, Validate

In [3]:
train_df = df[df["partition"] == "train"].copy()
df_test  = df[df["partition"] == "test"].copy()

# I create a validation set from the train set
df_train, df_val = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42,
    stratify=train_df["three_partition_label"]
)

label_to_idx = {
    "malignant": 0,
    "benign": 1,
    "non-neoplastic": 2
}
idx_to_label = {v: k for k, v in label_to_idx.items()}

df_train["y"] = df_train["three_partition_label"].map(label_to_idx)
df_val["y"]   = df_val["three_partition_label"].map(label_to_idx)
df_test["y"]  = df_test["three_partition_label"].map(label_to_idx)

df_train = df_train[["preprocessed_path", "y"]].copy()
df_val   = df_val[["preprocessed_path", "y"]].copy()
df_test  = df_test[["preprocessed_path", "y"]].copy()

print("Train:", len(df_train))
print("Val  :", len(df_val))
print("Test :", len(df_test))

Train: 7151
Val  : 795
Test : 2272


## Data Augmentations

In [4]:
# Standard ImageNet normalization (because we're using ImageNet-pretrained weights)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

INPUT_SIZE = 384

train_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),   # keep consistent sizing
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## CNN Model

In [5]:
NUM_CLASSES = 3  # malignant, benign, non-neoplastic

def build_model(num_classes: int = NUM_CLASSES):
    # Pretrained EfficientNetV2-S
    model = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )

    # Replace classifier for 3 classes
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

model_1 = build_model()
print(model_1.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 213MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=3, bias=True)
)


## Set Device

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## Setting the Weights

In [7]:
# counts for classes 0,1,2
counts = df_train["y"].value_counts().sort_index().values  # e.g. [malignant, benign, non-neoplastic]
weights = 1.0 / counts
weights = weights / weights.sum() * len(weights)  # normalize so avg weight ~ 1

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

In [8]:
counts = df_train["y"].value_counts().sort_index()
print("counts:", counts.to_dict())

weights = 1.0 / counts.values
weights = weights / weights.sum() * len(weights)
print("weights:", weights)

# sanity: higher weight should go to rarer class
for i,w in enumerate(weights):
    print(i, idx_to_label[i], "weight=", float(w))

counts: {0: 1058, 1: 950, 2: 5143}
weights: [1.29343756 1.44048099 0.26608146]
0 malignant weight= 1.293437557468759
1 benign weight= 1.4404809850546814
2 non-neoplastic weight= 0.2660814574765598


## Loading the Data with the DataLoader

In [9]:
class SkinCSV_Dataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rel_path = row["preprocessed_path"].replace("\\", "/")

        # Your Kaggle storage has preprocessed/preprocessed/..., so add ONE extra folder:
        img_path = os.path.join(self.root_dir, "preprocessed", rel_path)

        img = Image.open(img_path).convert("RGB")
        y = int(row["y"])

        if self.transform:
            img = self.transform(img)

        return img, y

## Hyperparameter Tuning

In [10]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from sklearn.metrics import f1_score

@torch.no_grad()
def eval_macro_f1(model, loader, device):
    model.eval()
    all_y, all_p = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = torch.argmax(logits, dim=1)
        all_y.append(y.cpu().numpy())
        all_p.append(preds.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    return f1_score(y_true, y_pred, average="macro")

def make_transforms(aug_strength: str):
    # Uses your INPUT_SIZE / IMAGENET_* and avoids RandomResizedCrop
    if aug_strength == "light":
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    elif aug_strength == "medium":
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.3),
            transforms.RandomRotation(20),
            transforms.ColorJitter(0.15, 0.15, 0.15, 0.03),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    else:  # strong
        train_transforms = transforms.Compose([
            transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.35),
            transforms.RandomRotation(25),
            RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3), value='random'),
        ])

    val_test_transforms = transforms.Compose([
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transforms, val_test_transforms

def make_balanced_sampler(df_train):
    class_counts = df_train["y"].value_counts().sort_index().to_dict()
    sample_w = df_train["y"].map(lambda c: 1.0 / class_counts[int(c)]).values
    sample_w = torch.DoubleTensor(sample_w)
    return WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

def unfreeze_last_blocks(model, n_blocks: int):
    for p in model.parameters():
        p.requires_grad = False
    if n_blocks > 0:
        for p in model.features[-n_blocks:].parameters():
            p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True

def train_epochs_amp(model, loader_train, loader_val, device, epochs, optimizer, criterion, scheduler=None):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for x, y in loader_train:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if scheduler is not None:
            scheduler.step()

        f1 = eval_macro_f1(model, loader_val, device)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_f1, best_state

def objective(trial):
    aug_strength = trial.suggest_categorical("aug_strength", ["light", "medium", "strong"])
    unfreeze_blocks = trial.suggest_int("unfreeze_blocks", 2, 8)

    head_lr = trial.suggest_float("head_lr", 3e-4, 3e-3, log=True)
    backbone_lr = trial.suggest_float("backbone_lr", 1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.05)
    BATCH_SIZE_trial = trial.suggest_categorical("BATCH_SIZE", [32, 48, 64])

    train_transforms, val_test_transforms = make_transforms(aug_strength)

    ds_train = SkinCSV_Dataset(df_train, DATA_DIR, transform=train_transforms)
    ds_val   = SkinCSV_Dataset(df_val,   DATA_DIR, transform=val_test_transforms)

    sampler = make_balanced_sampler(df_train)
    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE_trial, sampler=sampler, num_workers=2, pin_memory=True)
    loader_val   = DataLoader(ds_val,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model = build_model(num_classes=NUM_CLASSES).to(device)
    unfreeze_last_blocks(model, unfreeze_blocks)

    backbone_params, head_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if "classifier" in name:
            head_params.append(p)
        else:
            backbone_params.append(p)

    optimizer = optim.AdamW(
        [{"params": backbone_params, "lr": backbone_lr},
         {"params": head_params,     "lr": head_lr}],
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss(
        label_smoothing=label_smoothing
    )

    trial_epochs = 4  # keep short for search
    f1, _ = train_epochs_amp(model, loader_train, loader_val, device, trial_epochs, optimizer, criterion)

    trial.report(f1, step=trial_epochs)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return f1

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("Best macro-F1:", study.best_value)
print("Best params:", study.best_params)

[I 2026-02-14 18:06:55,316] A new study created in memory with name: no-name-9c33fc60-6531-4477-8978-df408f180840
/tmp/ipykernel_55/2049327602.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_55/2049327602.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/2049327602.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/2049327602.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device

Best macro-F1: 0.7029834074509145
Best params: {'aug_strength': 'light', 'unfreeze_blocks': 7, 'head_lr': 0.0004197486446879765, 'backbone_lr': 0.00025357950719166625, 'weight_decay': 2.192024256578768e-05, 'label_smoothing': 0.012139633080454966, 'BATCH_SIZE': 64}


## Model Building

In [11]:
BEST = study.best_params

train_transforms, val_test_transforms = make_transforms(BEST["aug_strength"])

ds_train = SkinCSV_Dataset(df_train, DATA_DIR, transform=train_transforms)
ds_val   = SkinCSV_Dataset(df_val,   DATA_DIR, transform=val_test_transforms)
ds_test  = SkinCSV_Dataset(df_test,  DATA_DIR, transform=val_test_transforms)

sampler = make_balanced_sampler(df_train)

loader_train = DataLoader(ds_train, batch_size=BEST["BATCH_SIZE"], sampler=sampler,
                          num_workers=2, pin_memory=True)
loader_val   = DataLoader(ds_val, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(ds_test, batch_size=64, shuffle=False,
                          num_workers=2, pin_memory=True)

model = build_model(num_classes=NUM_CLASSES).to(device)
unfreeze_last_blocks(model, BEST["unfreeze_blocks"])

backbone_params, head_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if "classifier" in name:
        head_params.append(p)
    else:
        backbone_params.append(p)

optimizer = optim.AdamW(
    [{"params": backbone_params, "lr": BEST["backbone_lr"]},
     {"params": head_params,     "lr": BEST["head_lr"]}],
    weight_decay=BEST["weight_decay"]
)

criterion = nn.CrossEntropyLoss(
    label_smoothing=BEST["label_smoothing"]
)

# Train longer than Optuna trials
EPOCHS = 20

# Simple cosine schedule (good default)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

'''best_f1, best_state = train_epochs_amp(
    model, loader_train, loader_val, device,
    epochs=EPOCHS,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler
)

print("Best VAL macro-F1 during training:", best_f1)

# Load best checkpoint
model.load_state_dict(best_state)

# Evaluate on TEST
test_f1 = eval_macro_f1(model, test_loader, device)
print("TEST macro-F1:", test_f1)'''

'best_f1, best_state = train_epochs_amp(\n    model, loader_train, loader_val, device,\n    epochs=EPOCHS,\n    optimizer=optimizer,\n    criterion=criterion,\n    scheduler=scheduler\n)\n\nprint("Best VAL macro-F1 during training:", best_f1)\n\n# Load best checkpoint\nmodel.load_state_dict(best_state)\n\n# Evaluate on TEST\ntest_f1 = eval_macro_f1(model, test_loader, device)\nprint("TEST macro-F1:", test_f1)'

In [12]:
'''model.load_state_dict(best_state)'''

'model.load_state_dict(best_state)'

## Mixup

In [13]:
import numpy as np
import torch
import torch.nn.functional as F

def mixup_batch(x, y, alpha=0.4):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha <= 0:
        return x, y, y, 1.0

    lam = np.random.beta(alpha, alpha)
    bs = x.size(0)
    idx = torch.randperm(bs, device=x.device)

    x_mix = lam * x + (1 - lam) * x[idx]
    y_a = y
    y_b = y[idx]
    return x_mix, y_a, y_b, lam

def mixup_criterion(logits, y_a, y_b, lam, class_weights=None, label_smoothing=0.0):
    # Use CE with label smoothing and optional class weights, applied to both targets
    loss_a = F.cross_entropy(logits, y_a, weight=class_weights, label_smoothing=label_smoothing)
    loss_b = F.cross_entropy(logits, y_b, weight=class_weights, label_smoothing=label_smoothing)
    return lam * loss_a + (1 - lam) * loss_b

In [14]:
def train_epochs_amp_mixup(model, loader_train, loader_val, device, epochs, optimizer,
                           class_weights, label_smoothing=0.0, mixup_alpha=0.4, scheduler=None):

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    best_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for x, y in loader_train:
            x = x.to(device)
            y = y.to(device)

            x, y_a, y_b, lam = mixup_batch(x, y, alpha=mixup_alpha)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                logits = model(x)
                loss = mixup_criterion(
                    logits, y_a, y_b, lam,
                    class_weights=class_weights,
                    label_smoothing=label_smoothing
                )

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        if scheduler is not None:
            scheduler.step()

        f1 = eval_macro_f1(model, loader_val, device)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    return best_f1, best_state

## Evaluate Functions

In [15]:
import numpy as np
import torch

@torch.no_grad()
def eval_metrics(model, loader, device, num_classes=NUM_CLASSES):
    model.eval()

    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64)
    total = 0
    correct = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == y).sum().item()
        total += y.numel()

        # confusion matrix update
        for t, p in zip(y.view(-1), preds.view(-1)):
            cm[t.long(), p.long()] += 1

    acc = correct / total if total > 0 else 0.0

    # recall per class: TP / (TP + FN) = diag / row_sum
    row_sums = cm.sum(dim=1).clamp(min=1)
    recall = (cm.diag().float() / row_sums.float()).cpu().numpy()

    return acc, cm.cpu(), recall

def print_eval_report(name, acc, cm, recall, class_names=None):
    print(f"\n{name} accuracy: {acc*100:.2f}%")
    print(f"{name} confusion matrix (rows=true, cols=pred):\n{cm}")

    if class_names is None:
        class_names = [f"class_{i}" for i in range(len(recall))]

    print(f"{name} recall per class:")
    for i, r in enumerate(recall):
        print(f"  {class_names[i]}: {r:.4f}")

In [16]:
mixup_alpha = 0.15  # good starting point

best_f1, best_state = train_epochs_amp_mixup(
    model=model,
    loader_train=loader_train,
    loader_val=loader_val,
    device=device,
    epochs=EPOCHS,
    optimizer=optimizer,
    class_weights=None,
    label_smoothing=BEST["label_smoothing"],
    mixup_alpha=mixup_alpha,
    scheduler=scheduler
)

model.load_state_dict(best_state)

/tmp/ipykernel_55/1230852955.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
/tmp/ipykernel_55/1230852955.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is depreca

<All keys matched successfully>

In [17]:
import torch
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

@torch.no_grad()
def eval_classification_report(model, loader, device, class_names):
    model.eval()

    y_true = []
    y_pred = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        preds = torch.argmax(logits, dim=1)

        y_true.append(y.cpu().numpy())
        y_pred.append(preds.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    print("\nAccuracy:", accuracy_score(y_true, y_pred))
    print("\nConfusion Matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )


In [18]:
eval_classification_report(
    model,
    test_loader,
    device,
    class_names=["malignant", "benign", "non-neoplastic"]
)


Accuracy: 0.8397887323943662

Confusion Matrix (rows=true, cols=pred):
[[ 203   22  113]
 [  16  139  145]
 [  27   41 1566]]

Classification Report:
                precision    recall  f1-score   support

     malignant     0.8252    0.6006    0.6952       338
        benign     0.6881    0.4633    0.5538       300
non-neoplastic     0.8586    0.9584    0.9057      1634

      accuracy                         0.8398      2272
     macro avg     0.7906    0.6741    0.7182      2272
  weighted avg     0.8311    0.8398    0.8279      2272



In [19]:
@torch.no_grad()
def get_probs(model, loader, device):
    model.eval()
    probs_all = []
    y_all = []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu()
        probs_all.append(probs)
        y_all.append(y)
    return torch.cat(probs_all).numpy(), torch.cat(y_all).numpy()

In [20]:
from sklearn.metrics import recall_score

val_probs, val_y = get_probs(model, loader_val, device)

best_tm, best_tb, best_score = None, None, -1

# You can widen ranges later; start tight
for t_m in np.linspace(0.20, 0.60, 41):
    for t_b in np.linspace(0.20, 0.60, 41):
        pred = np.argmax(val_probs, axis=1)

        # Force malignant / benign when prob crosses threshold
        pred = np.where(val_probs[:,0] >= t_m, 0, pred)  # malignant
        pred = np.where(val_probs[:,1] >= t_b, 1, pred)  # benign

        # objective: maximize average recall of malignant+benign
        r_m = recall_score(val_y == 0, pred == 0, zero_division=0)
        r_b = recall_score(val_y == 1, pred == 1, zero_division=0)
        score = 0.5 * (r_m + r_b)

        if score > best_score:
            best_score = score
            best_tm, best_tb = t_m, t_b

print("Best thresholds:", "t_m=", best_tm, "t_b=", best_tb)
print("Best VAL avg recall (malignant+benign):", best_score)

Best thresholds: t_m= 0.2 t_b= 0.2
Best VAL avg recall (malignant+benign): 0.550040355125101


In [21]:
test_probs, test_y = get_probs(model, test_loader, device)

pred = np.argmax(test_probs, axis=1)
pred = np.where(test_probs[:,0] >= best_tm, 0, pred)
pred = np.where(test_probs[:,1] >= best_tb, 1, pred)

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print("Accuracy:", accuracy_score(test_y, pred))
print("Confusion Matrix:\n", confusion_matrix(test_y, pred))
print(classification_report(
    test_y, pred,
    target_names=["malignant","benign","non-neoplastic"],
    digits=4, zero_division=0
))

Accuracy: 0.8362676056338029
Confusion Matrix:
 [[ 215   31   92]
 [  15  171  114]
 [  39   81 1514]]
                precision    recall  f1-score   support

     malignant     0.7993    0.6361    0.7084       338
        benign     0.6042    0.5700    0.5866       300
non-neoplastic     0.8802    0.9266    0.9028      1634

      accuracy                         0.8363      2272
     macro avg     0.7612    0.7109    0.7326      2272
  weighted avg     0.8317    0.8363    0.8321      2272

